Assignment: Gesture Classifier with Noise & Real-Time Test
Scenario

You have a glove with 11 sensors (like your project). Each sensor gives a value in real-time. Your goal is to train a small neural network that is robust to sensor noise, evaluate it, and simulate a real-time prediction.

Step 1 — Synthetic Dataset
Instead of Iris, create a dataset with 5 gestures (labels 0–4).
Each gesture has 11 sensor readings. For simplicity:
# Simulate dataset

```
import torch
import numpy as np
num_samples = 1000
num_sensors = 11
num_classes = 5

X = np.random.rand(num_samples, num_sensors) * 100  # sensor values 0-100
Y = np.random.randint(0, num_classes, size=num_samples)

```



Step 2 — Add noise

Simulate real-world sensor noise
```
noise = np.random.normal(0, 5, X.shape)  # mean=0, std=5
X_noisy = X + noise
Step 3 — Train a small NN
Input: 11 features
Hidden layers: 16 → 12
Output: 5 classes
```
Use ReLU and CrossEntropyLoss
Step 4 — Evaluate accuracy
Split train/test 80/20
Print accuracy like you did before
Step 5 — Real-time simulation
Pick 5 random new “sensor readings”, predict the gesture, and print it.
```
sample = torch.tensor(np.random.rand(1,11)*100, dtype=torch.float32)
prediction = torch.argmax(Model(sample)).item()
print(f"Predicted gesture: {prediction}")
```
Step 6 — Challenge / Extra Credit
Robustness: Add dropout to your NN and show the model is more resistant to noisy readings.
```
Weight export: Save model weights to a .pt file.
```
Later you can use these weights to implement the network manually on ESP32.
Visualization: Plot the predicted gestures vs actual gestures in a confusion matrix.

In [ ]:
import torch

from torch.nn.modules.linear import Linear
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_iris
import torch.nn as nn
import torch.optim as optim

num_sensors=11
num_samples=1000
num_classes=5
X=np.random.rand(num_samples,num_sensors)*100
Y=np.random.randint(0,num_classes,size=num_samples)

XNoise=np.random.normal(0,5,X.shape)
print(Y.shape)


(1000,)


In [ ]:
class GestureClassifier(nn.Module):

   def __init__(self,num_classes,num_sensors):
    super().__init__()
    self.num_classes=num_classes
    self.num_sensors=num_sensors


    self.model=nn.Sequential(nn.Linear(11,16),
                             nn.ReLU(),
                             nn.Linear(16,12),
                             nn.ReLU(),

                             nn.Linear(12,self.num_classes))
   def forward(self,inputs):
     return self.model(inputs)



In [ ]:
# Gesture 0 → low values
X0 = np.random.normal(20, 5, (200, 11))

# Gesture 1 → high values
X1 = np.random.normal(80, 5, (200, 11))

# Gesture 2 → increasing pattern
X2 = np.tile(np.linspace(10, 90, 11), (200,1)) + np.random.normal(0,5,(200,11))

# Gesture 3 → decreasing pattern
X3 = np.tile(np.linspace(90, 10, 11), (200,1)) + np.random.normal(0,5,(200,11))

# Gesture 4 → mixed/random but centered
X4 = np.random.normal(50, 10, (200, 11))


X=np.vstack([X0,X1,X2,X3,X4])
Y0 = np.zeros(200)
Y1 = np.ones(200)
Y2 = np.full(200, 2)
Y3 = np.full(200, 3)
Y4 = np.full(200, 4)
Y = np.concatenate([Y0, Y1, Y2, Y3, Y4])
X_train, X_test, y_train, y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.long)
print(y_test.shape)


torch.Size([200])


In [ ]:
model=GestureClassifier(num_classes=num_classes,num_sensors=num_sensors)
criterion=nn.CrossEntropyLoss()

optimizer=optim.AdamW(model.parameters(),lr=0.01)

epochs=200

for i in range(epochs):
  model.train()
  output=model.forward(X_train)
  loss=criterion(output,y_train)

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()

  if (i + 1) % 10 == 0:
        print(f"Epoch {i+1}, Loss: {loss.item():.4f}")


Epoch 10, Loss: 0.8866
Epoch 20, Loss: 0.6262
Epoch 30, Loss: 0.5559
Epoch 40, Loss: 0.4979
Epoch 50, Loss: 0.4525
Epoch 60, Loss: 0.4173
Epoch 70, Loss: 0.3847
Epoch 80, Loss: 0.3550
Epoch 90, Loss: 0.3276
Epoch 100, Loss: 0.3010
Epoch 110, Loss: 0.2754
Epoch 120, Loss: 0.2513
Epoch 130, Loss: 0.2286
Epoch 140, Loss: 0.2080
Epoch 150, Loss: 0.1900
Epoch 160, Loss: 0.1693
Epoch 170, Loss: 0.1537
Epoch 180, Loss: 0.1374
Epoch 190, Loss: 0.1216
Epoch 200, Loss: 0.1090


In [ ]:
model.eval()

with torch.no_grad():
  outputs=model(X_test)
  _,predicted=torch.max(outputs,1)
  accuracy=(predicted==y_test).float().mean()

  print(f"accuracy={accuracy.item():.4f}")

accuracy=0.9150
